# Visualizaciones: perfil sonoro y artistas por país

Este notebook arma los gráficos del proyecto. Las tablas ya están hechas en `03_tablas.ipynb`
(carpeta `data/tablas/`), así que acá solo las leemos y las graficamos.

Cada pregunta sigue la secuencia de la consigna: **pregunta y tarea → datos → transformación →
mapeo visual y representación → interacción → interpretación**.

## Contenido
0. Preparación del entorno y carga de tablas
1. Pregunta 1 (geoespacial): ¿cambia el perfil sonoro entre países?
2. Pregunta 2 (relacional): ¿qué países se parecen en los artistas que escuchan?
3. Pregunta 3 (temporal): ¿cómo cambió el ranking de artistas entre 2005 y 2014?
4. Pregunta 4 (multidimensional): ¿el perfil sonoro se relaciona con la cultura y la felicidad?
5. Pregunta 5 (multidimensional, comparar grupos): ¿cambia el perfil sonoro con la edad?
6. Resumen de preguntas, gráficos e interacciones

## 0. Preparación del entorno

Las rutas, la conexión a DuckDB y las funciones `q()` y `guardar()` están en `config.py`, que tiene
que estar en la misma carpeta que este notebook. Para los gráficos usamos **plotly** (interactivos)
y matplotlib (estáticos). Para la pregunta 2 usamos `scipy` y `scikit-learn`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go          # si da error, instalar con: pip install plotly
import plotly.express as px
import pycountry                           # para pasar el código de 2 letras al de 3 que usa plotly

from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from config import q, ruta_tablas, FEATURES, MIN_USUARIOS_PAIS

print("Features:", FEATURES)

Features: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'tempo_norm', 'loudness_norm']


### Carga de tablas

In [2]:
paises = q(f"SELECT * FROM '{ruta_tablas}tabla_pais.parquet'")                   # pregunta 1
pais_artista = q(f"SELECT * FROM '{ruta_tablas}pais_artista.parquet'")           # pregunta 2
artista_anio = q(f"SELECT * FROM '{ruta_tablas}artista_anio.parquet'")           # pregunta 3
cultura = q(f"SELECT * FROM '{ruta_tablas}tabla_pais_cultura.parquet'")          # pregunta 4
edad = q(f"SELECT * FROM '{ruta_tablas}tabla_edad.parquet'")                     # pregunta 5

for nombre, tabla in [("tabla_pais", paises), ("pais_artista", pais_artista), ("artista_anio", artista_anio),
                      ("tabla_pais_cultura", cultura), ("tabla_edad", edad)]:
    print(f"{nombre}: {len(tabla):,} filas")

tabla_pais: 56 filas
pais_artista: 458,744 filas
artista_anio: 300 filas
tabla_pais_cultura: 47 filas
tabla_edad: 4 filas


### Nombre y código de 3 letras de cada país

Plotly ubica los países en el mapa con el código de **3 letras** (ISO-3), y nosotros tenemos el de 2.
Con `pycountry` creamos una columna con el código de 3 letras y otra con el nombre, que usamos en el
mapa y en los demás gráficos. Es un atributo derivado.

In [3]:
def buscar_pais(codigo):
    # Devuelve el código de 3 letras y el nombre del país, o (None, None) si no lo encuentra
    try:
        pais = pycountry.countries.get(alpha_2=codigo)
        return pais.alpha_3, pais.name
    except (AttributeError, KeyError):
        return None, None

# apply() aplica la función a cada código; zip(*...) separa los dos valores en dos columnas
paises["iso3"], paises["nombre"] = zip(*paises["country"].apply(buscar_pais))

# Revisamos que ningún país se haya quedado sin código (si aparece alguno, no se vería en el mapa)
print("Países sin código de 3 letras:", paises[paises["iso3"].isna()]["country"].tolist())

# Diccionario código de 2 letras -> nombre, para usar en los demás gráficos
nombre_de = dict(zip(paises["country"], paises["nombre"]))

Países sin código de 3 letras: []


## 1. Pregunta 1: ¿el perfil sonoro de la música cambia de un país a otro?

**Pregunta.** ¿La música que escuchan los usuarios suena distinto según el país? 

**Tareas**
- **Comparar** países entre sí.
- **Localizar extremos:** qué países están más arriba y más abajo en cada feature.
- **Detectar patrones:** si países vecinos se parecen.

**Enfoque:** geoespacial, porque el país se puede ubicar en un mapa.

**Abstracción de los datos** (`tabla_pais`)

| | Qué es | Tipo |
|---|---|---|
| Ítem | un país | |
| `country` | código del país | Categórico |
| `n_usuarios` | cantidad de usuarios del país | Cuantitativo |
| `danceability`, `energy`, `valence`, `acousticness`, `instrumentalness`, `speechiness`, `liveness`, `tempo_norm`, `loudness_norm` | promedio de la feature en ese país | Cuantitativo (entre 0 y 1) |



### 1.2 Transformación

La tabla `tabla_pais` se armó en `03_tablas.ipynb` con dos operaciones: **agregación** (promedio de
cada feature por país; cada usuario ya viene con su propio promedio, así que cuenta una sola vez) y
**filtrado** (solo países con al menos `MIN_USUARIOS_PAIS` usuarios). Los países que quedan afuera
aparecen en gris en el mapa. Acá agregamos el código de 3 letras y el nombre.

Antes de graficar miramos qué tan distintos son los países entre sí. Si los promedios casi no cambian,
hay que tenerlo en cuenta al elegir la escala de color.

In [4]:
print("Países:", len(paises), "| Usuarios que representan:", f"{paises['n_usuarios'].sum():,}")

resumen = paises[FEATURES].agg(["min", "max"]).T
resumen["diferencia"] = resumen["max"] - resumen["min"]
resumen.round(3)

Países: 56 | Usuarios que representan: 52,098


,min,max,diferencia
danceability,0.443,0.537,0.094
energy,0.578,0.741,0.163
valence,0.308,0.462,0.155
acousticness,0.152,0.331,0.179
instrumentalness,0.120,0.319,0.199
speechiness,0.061,0.082,0.021
liveness,0.184,0.210,0.027
tempo_norm,0.412,0.434,0.022
loudness_norm,0.754,0.794,0.040


### 1.3 Mapeo visual y representación

| Qué mostramos | Marca | Canal | Por qué |
|---|---|---|---|
| El país | Área (el polígono del país) | Posición, la da el mapa | Es un dato geográfico y queremos ver vecinos y regiones |
| Valor de la feature | | **Color**, escala secuencial de una sola variable | Es cuantitativo y ordenado, y una escala de claro a oscuro respeta ese orden |
| Cantidad de usuarios | | Texto en el tooltip | Sirve para saber cuánto confiar en el promedio, sin llenar el mapa |
| Países sin datos | | Gris | Que no se confundan con un país de valor bajo |

**Decisiones:**
- **Mapa plano.** La tarea principal es comparar países
- **Escala de color de mínimo a máximo.** Como los promedios son parecidos entre países, con una escala de 0 a 1 todos tendrían casi el mismo color. Cada feature usa su propio
  mínimo y máximo
- **Color y exactitud.** El color no permite leer valores con precisión.

**Interacción.** Un **selector de feature**: la tarea es comparar el mismo mapa para distintos atributos
sin hacer nueve mapas. Al pasar el mouse por un país aparece su nombre, su valor y cuántos usuarios
tiene, para **identificar** un país y saber si el dato es confiable.

### 1.4 Mapa interactivo

In [5]:
primera = FEATURES[0]

# Datos que se ven al pasar el mouse: nombre y cantidad de usuarios
info = paises[["nombre", "n_usuarios"]].values

mapa = go.Figure(go.Choropleth(
    locations=paises["iso3"],                 # qué país es (código de 3 letras)
    z=paises[primera],                        # valor que se pinta
    zmin=paises[primera].min(),               # la escala va del mínimo al máximo de los países
    zmax=paises[primera].max(),
    colorscale="Viridis",
    colorbar=dict(title=primera),
    customdata=info,
    hovertemplate="<b>%{customdata[0]}</b><br>valor: %{z:.3f}<br>usuarios: %{customdata[1]:,}<extra></extra>",
))

# Un botón por feature. Al apretarlo cambian los valores, la escala, el título de la barra y del gráfico.
botones = []
for f in FEATURES:
    botones.append(dict(
        label=f,
        method="update",
        args=[
            {"z": [paises[f].tolist()], "zmin": [paises[f].min()], "zmax": [paises[f].max()],
             "colorbar.title.text": f},
            {"title.text": f"Promedio de {f} por país"},
        ],
    ))

mapa.update_layout(
    title=f"Promedio de {primera} por país",
    updatemenus=[dict(type="dropdown", buttons=botones, x=0.01, y=0.98,
                      xanchor="left", yanchor="top")],
    height=550,
    margin=dict(l=0, r=0, t=60, b=0),
)

# Mapa plano; los países sin datos quedan en gris
mapa.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False,
                 showland=True, landcolor="lightgray")

mapa.show()

**Versión ampliada (opcional).** Con `05_datos_mapa_web.ipynb` se generan los datos de
`mapa_web/mapa_paises.html`, un mapa con filtros por género y edad, y un panel por país con su perfil
contra el promedio mundial y sus artistas más escuchados. 

## 3. ¿cómo cambió el ranking de artistas entre 2005 y 2014?

**Pregunta.** ¿Los artistas más escuchados son siempre los mismos o hay cambios a lo largo de los años?

**Tareas:** detectar **tendencias** (artistas que suben o bajan) y **cambios** (entradas y salidas del top).

**Enfoque:** temporal.

**Abstracción de los datos** (`artista_anio`)

| | Qué es | Tipo |
|---|---|---|
| Ítem | un artista en un año | |
| `anio` | año de la escucha (2005 a 2014) | Ordinal / temporal |
| `artist_name` | artista | Categórico |
| `puesto` | posición en el ranking del año (1 = más oyentes) | Ordinal |
| `oyentes` | usuarios distintos que lo escucharon ese año | Cuantitativo |

### 3.2 Transformación específica

La tabla ya trae el top 30 de cada año. Para el gráfico:
- **Filtrado:** nos quedamos con los artistas que estuvieron en el **top 10** en algún año.
- Para esos artistas usamos también sus puestos del 11 al `PUESTO_MAX`, así se ve cuando bajan o suben
  un poco fuera del top 10 y no se cortan de golpe.
- Los años en que un artista queda fuera de `PUESTO_MAX` quedan como hueco en la línea.

Hay que tener en cuenta que 2005 empieza en febrero y 2014 termina en septiembre, y que en los primeros
años hay menos usuarios activos. Por eso el ranking se compara por **puesto** y no por cantidad de oyentes.

In [6]:
PUESTO_MAX = 15

artista_anio["anio"] = artista_anio["anio"].astype(int)
anios = sorted(artista_anio["anio"].unique())

# Artistas que estuvieron en el top 10 al menos un año
en_top10 = artista_anio[artista_anio["puesto"] <= 10]["artist_id"].unique()
print("Artistas que alguna vez estuvieron en el top 10:", len(en_top10))

datos_bump = artista_anio[artista_anio["artist_id"].isin(en_top10) & (artista_anio["puesto"] <= PUESTO_MAX)]

# Los diez primeros de cada año, para mirar la tabla
artista_anio[artista_anio["puesto"] <= 10].pivot(index="puesto", columns="anio", values="artist_name")

Artistas que alguna vez estuvieron en el top 10: 26


anio,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
puesto,,,,,,,,,,
1,The White Stripes,The White Stripes,Radiohead,Radiohead,Radiohead,Radiohead,Radiohead,Radiohead,Daft Punk,Daft Punk
2,Radiohead,Radiohead,The White Stripes,Coldplay,Kings of Leon,Kings of Leon,Foo Fighters,Adele,Radiohead,Arctic Monkeys
3,System of a Down,System of a Down,Foo Fighters,The Killers,Coldplay,Coldplay,Daft Punk,Coldplay,Arctic Monkeys,Coldplay
4,Foo Fighters,Coldplay,The Killers,The Cure,The Killers,Daft Punk,Coldplay,Foo Fighters,The xx,Radiohead
5,Coldplay,Death Cab for Cutie,Arctic Monkeys,Foo Fighters,The Cure,The Cure,Adele,The xx,The Cure,The xx
6,Weezer,The Cure,System of a Down,System of a Down,Foo Fighters,The Killers,Kings of Leon,The Cure,Coldplay,The Cure
7,Gorillaz,The Postal Service,The Shins,Guns N' Roses,System of a Down,System of a Down,System of a Down,Arctic Monkeys,Foo Fighters,Foo Fighters
8,The Shins,Gorillaz,The Cure,Daft Punk,Arctic Monkeys,Gorillaz,Arctic Monkeys,Daft Punk,The Killers,Arcade Fire
9,The Cure,Foo Fighters,The Strokes,Beck,MGMT,MGMT,Red Hot Chili Peppers,Kings of Leon,Red Hot Chili Peppers,Lana Del Rey


### 3.3 Mapeo visual y representación

**Gráfico de ranking (bump chart)**

| Qué mostramos | Marca | Canal | Comentario |
|---|---|---|---|
| Tiempo | | Posición horizontal | El tiempo va en el eje x, de izquierda a derecha |
| Puesto en el ranking | Punto | Posición vertical |  |
| Un artista a lo largo de los años | Línea que une los puntos | Color | La línea permite seguir a un mismo artista |

**Decisiones:**
- Decisión. Para cada año, ordenamos a los artistas según la cantidad de usuarios distintos que los escucharon. Mostramos el puesto porque nos interesa ver qué artistas subieron, bajaron o se mantuvieron entre los primeros. En 2005 y 2014, el ranking corresponde solo a los meses disponibles.
- Con tantos artistas los colores se repiten y es difícil seguir una línea, por eso agregamos interacción.
- Una línea punteada marca el límite del top 10.

**Interacción:** Al elegir un artista, su línea se resalta y las demás quedan en segundo plano. Así es más fácil seguir cómo cambió su puesto año a año. La tarea es seguir la
trayectoria de un artista concreto sin perderla entre las otras. Al pasar el mouse se ve el año, el puesto y
la cantidad de oyentes. Con un click en la leyenda también se puede ocultar un artista.

### 3.4 Evolución del ranking

In [15]:
import altair as alt

datos = datos_bump.sort_values(["artist_id", "anio"]).copy()

# Si el artista sale del ranking y vuelve después, empieza una línea nueva.
# Así evitamos unir dos años separados por un período fuera del top 15.
datos["tramo"] = datos.groupby("artist_id")["anio"].transform(
    lambda anios: anios.diff().ne(1).cumsum()
)

nombres = sorted(datos["artist_name"].unique())
selector = alt.param(
    name="artista_elegido",
    value="Todos",
    bind=alt.binding_select(options=["Todos"] + nombres, name="Artista: ")
)

# El puesto 1 va arriba y los años se muestran sin decimales.
base = alt.Chart(datos).encode(
    x=alt.X("anio:Q", title="Año",
            axis=alt.Axis(values=list(anios), format="d", grid=False),
            scale=alt.Scale(nice=False, zero=False)),
    y=alt.Y("puesto:Q", title="Puesto por cantidad de oyentes",
            scale=alt.Scale(domain=[PUESTO_MAX + 0.5, 0.5]),
            axis=alt.Axis(values=list(range(1, PUESTO_MAX + 1)))),
    detail=["artist_id:N", "tramo:N"],
    order="anio:Q"
)

# Las líneas grises dejan ver dónde estaban los demás artistas.
fondo = base.mark_line(
    color="#cbd1d8", strokeWidth=1.3,
    point=alt.OverlayMarkDef(size=16, filled=True)
)

# La línea elegida se dibuja encima para que se pueda seguir en los cruces.
visibles = base.transform_filter(
    (selector == "Todos") | (alt.datum.artist_name == selector)
)

lineas = visibles.mark_line(strokeWidth=2.8).encode(
    color=alt.condition(
        selector == "Todos",
        alt.Color("artist_name:N", legend=None, scale=alt.Scale(scheme="tableau20")),
        alt.value("#2457a7")
    )
)

puntos = visibles.mark_circle(size=45).encode(
    color=lineas.encoding.color,
    tooltip=[
        alt.Tooltip("artist_name:N", title="Artista"),
        alt.Tooltip("anio:Q", title="Año", format="d"),
        alt.Tooltip("puesto:Q", title="Puesto", format="d"),
        alt.Tooltip("oyentes:Q", title="Cant escuchas", format=",d")
    ]
)

# Etiquetamos el último punto visible del artista, aunque sea anterior a 2014.
etiqueta = (
    base.transform_filter(alt.datum.artist_name == selector)
    .transform_window(
        fila="row_number()",
        sort=[alt.SortField("anio", order="descending")]
    )
    .transform_filter(alt.datum.fila == 1)
    .transform_calculate(texto="datum.artist_name + ' · ' + datum.puesto + '°'")
    .mark_text(align="left", dx=8, fontSize=13, fontWeight="bold", color="#2457a7")
    .encode(text="texto:N")
)

limite_top10 = alt.Chart().mark_rule(
    color="#999999", strokeDash=[4, 4]
).encode(y=alt.datum(10.5))

bump = (
    alt.layer(fondo, limite_top10, lineas, puntos, etiqueta)
    .add_params(selector)
    .properties(
        width=680, height=380,
        title="¿Cómo cambió el ranking de artistas con más oyentes?"
    )
    .configure_view(stroke=None)
    .configure_axis(gridColor="#eeeeee", domain=False, labelFontSize=11)
)

bump

alt.LayerChart(...)

In [16]:
permanencia = (
    artista_anio.loc[artista_anio["puesto"] <= 10]
    .groupby(["artist_id", "artist_name"], as_index=False)
    .agg(anios_top10=("anio", "nunique"))
    .sort_values(["anios_top10", "artist_name"], ascending=[False, True])
)

alt.Chart(permanencia).mark_bar(color="#4575a5").encode(
    x=alt.X("anios_top10:Q", title="Cantidad de años en el top 10",
            axis=alt.Axis(tickMinStep=1)),
    y=alt.Y("artist_name:N", title=None, sort="-x"),
    tooltip=["artist_name:N", "anios_top10:Q"]
).properties(title="¿Cuántos años estuvo cada artista en el top 10?")

alt.Chart(...)

In [18]:
alcance = q("""
    WITH activos AS (
        SELECT
            YEAR(TO_TIMESTAMP("timestamp")) AS anio,
            COUNT(DISTINCT user_id) AS usuarios_activos
        FROM events_clean
        GROUP BY anio
    ),
    por_artista AS (
        SELECT
            YEAR(TO_TIMESTAMP(e."timestamp")) AS anio,
            a.artist_name AS artista,
            COUNT(DISTINCT e.user_id) AS oyentes
        FROM events_clean e
        JOIN artists a ON e.artist_id = a.artist_id
        WHERE a.artist_name IN ('Radiohead', 'Daft Punk', 'The White Stripes')
        GROUP BY anio, a.artist_name
    )
    SELECT
        p.anio, p.artista, p.oyentes, a.usuarios_activos,
        p.oyentes * 1.0 / a.usuarios_activos AS proporcion
    FROM por_artista p
    JOIN activos a ON p.anio = a.anio
    ORDER BY p.artista, p.anio
""")

grafico = alt.Chart(alcance).mark_line(
    point=True, strokeWidth=3
).encode(
    x=alt.X("anio:O", title="Año", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("proporcion:Q", title="Porcentaje de usuarios activos",
            axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=True)),
    color=alt.Color(
        "artista:N", title=None,
        scale=alt.Scale(
            domain=["The White Stripes", "Radiohead", "Daft Punk"],
            range=["#ba4436", "#3868ad", "#27816b"]
        ),
        legend=alt.Legend(orient="bottom")
    ),
    tooltip=[
        alt.Tooltip("artista:N", title="Artista"),
        alt.Tooltip("anio:O", title="Año"),
        alt.Tooltip("proporcion:Q", title="Usuarios que lo escucharon", format=".1%"),
        alt.Tooltip("oyentes:Q", title="Oyentes distintos", format=",")
    ]
).properties(
    title="¿Qué porcentaje de los usuarios escuchó a cada artista?",
    width=620, height=300
).configure_view(stroke=None)

grafico

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

Entre 2005 y 2014, 26 artistas pasaron por el top 10, pero solo tres llegaron al primer puesto. The White Stripes encabezó el ranking en 2005 y 2006. Después, Radiohead se mantuvo primero durante seis años seguidos, de 2007 a 2012. Daft Punk ocupó el primer lugar en 2013 y también en los meses registrados de 2014.

## 4. Pregunta 4: ¿el perfil sonoro se relaciona con la cultura y la felicidad de un país?


**Pregunta.** ¿Los países donde la gente es más individualista, o más feliz, escuchan música con
características distintas (por ejemplo, más alegre o más energética)?


## 5. Pregunta 5: 5. ¿Qué diferencias hay entre las franjas de edad en las características de la música que escuchan los usuarios?

**Tarea:** **comparar grupos** (franjas de edad) en varias variables a la vez.

**Enfoque:** multidimensional.

**Abstracción de los datos** (`tabla_edad`, una franja por fila)

| Atributo | Qué es | Tipo |
|---|---|---|
| `franja_edad` | 10-19, 20-24, 25-29, 30+ | **Ordinal** (tiene orden) |
| `n_usuarios` | usuarios de la franja | Cuantitativo |
| features | promedio de la feature en la franja | Cuantitativo (entre 0 y 1) |

### 5.2 Transformación específica

Las franjas se armaron en `03_tablas.ipynb` (derivación a partir de la edad) y los usuarios sin edad
válida quedaron afuera. Para el **radar** usamos solo 7 features.
Se pueden cambiar editando `EJES_RADAR`.

In [9]:
EJES_RADAR = ["danceability", "energy", "valence", "acousticness", "speechiness", "tempo_norm", "loudness_norm"]

valores = edad.set_index("franja_edad")[FEATURES]
print(edad[["franja_edad", "n_usuarios"]].to_string(index=False))

# Cuánto cambia cada feature entre franjas: diferencia entre la franja más alta y la más baja
valores.agg(["min", "max"]).T.assign(diferencia=lambda d: d["max"] - d["min"]).round(3)

franja_edad  n_usuarios
      10-19        8061
      20-24       17001
      25-29        8556
        30+        7710


,min,max,diferencia
danceability,0.489,0.509,0.020
energy,0.649,0.733,0.084
valence,0.416,0.441,0.025
acousticness,0.171,0.255,0.085
instrumentalness,0.165,0.234,0.069
speechiness,0.070,0.082,0.012
liveness,0.199,0.207,0.008
tempo_norm,0.420,0.433,0.013
loudness_norm,0.767,0.794,0.028


### 5.3 Mapeo visual y representación

**Gráfico de radar**

| Qué mostramos | Marca | Canal | Por qué |
|---|---|---|---|
| Una franja de edad | Polígono (línea cerrada) | **Color** (categórico) | Cada franja es una categoría; se puede ver su "forma" completa |
| Feature | | Posición angular (un eje por feature) | Permite comparar el perfil de cada grupo en varias variables a la vez |
| Valor | | Distancia al centro | Más lejos del centro, más alto dentro de ese eje |


**Interacción:** en la **leyenda** se activa y desactiva cada franja, para comparar dos grupos a la vez
sin que los cuatro polígonos se tapen. Al pasar el mouse sobre un punto aparece el valor real.

### 5.4 Radar por franja de edad

In [10]:
# Cada eje va de 0 (franja con el valor más bajo) a 1 (franja con el valor más alto)
sub = valores[EJES_RADAR]
escalado = (sub - sub.min()) / (sub.max() - sub.min())

radar = go.Figure()
for franja in sub.index:
    # Repetimos el primer punto al final para que el polígono quede cerrado
    ejes = EJES_RADAR + [EJES_RADAR[0]]
    r = escalado.loc[franja].tolist() + [escalado.loc[franja].iloc[0]]
    reales = sub.loc[franja].tolist() + [sub.loc[franja].iloc[0]]
    radar.add_trace(go.Scatterpolar(
        r=r, theta=ejes, name=franja, fill="toself", opacity=0.45,
        customdata=reales,
        hovertemplate="<b>" + franja + "</b><br>%{theta}: %{customdata:.3f}<extra></extra>",
    ))

radar.update_layout(
    title="Perfil sonoro por franja de edad (cada eje ajustado a su propio rango)",
    polar=dict(radialaxis=dict(range=[0, 1], showticklabels=False)),
    height=600,
)
radar.show()